In [ ]:
%pip install yfinance pandas numpy statsmodels pmdarima prophet neuralprophet neuralforecast fastapi uvicorn redis dash plotly

In [ ]:
import sys
import subprocess

# Force install yfinance into the exact running Python executable
subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "pyarrow", "fastparquet"])

# Refresh module paths
import site
from importlib import reload
reload(site)

# Now import yfinance
import yfinance as yf
print("yfinance successfully loaded! Version:", yf.__version__)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import os

TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'JPM', 'JNJ', 'V', 'PG']

def fetch_and_process_data():
    print("[1/3] Downloading 5 years OHLCV data via yfinance...")
    # Threads=False avoids SQLite database lock errors in yfinance
    raw = yf.download(TICKERS, period="5y", interval="1d", threads=False)
    
    # Reshape MultiIndex dataframe to long format
    df = raw.stack(level=1, future_stack=True).reset_index()
    df.columns.name = None
    df.rename(columns={'Ticker': 'unique_id', 'Date': 'ds', 'Close': 'y'}, inplace=True)
    df['ds'] = pd.to_datetime(df['ds'])
    
    # Drop any tickers that failed to download or have null values
    df = df.dropna(subset=['y'])
    df = df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

    print("[2/3] Engineering technical features & handling calendar gaps...")
    # Calculate daily returns
    df['returns'] = df.groupby('unique_id')['y'].pct_change()
    
    # Simple Moving Averages
    df['sma_20'] = df.groupby('unique_id')['y'].transform(lambda x: x.rolling(20).mean())
    df['sma_50'] = df.groupby('unique_id')['y'].transform(lambda x: x.rolling(50).mean())
    
    # Rolling Volatility (20-day standard deviation)
    df['volatility_20'] = df.groupby('unique_id')['returns'].transform(lambda x: x.rolling(20).std())

    # Relative Strength Index (RSI - 14 Days)
    delta = df.groupby('unique_id')['y'].diff()
    gain = (delta.where(delta > 0, 0)).groupby(df['unique_id']).transform(lambda x: x.rolling(14).mean())
    loss = (-delta.where(delta < 0, 0)).groupby(df['unique_id']).transform(lambda x: x.rolling(14).mean())
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))

    # Drop warm-up NaN rows from rolling indicators
    df.dropna(inplace=True)

    # Save to disk
    os.makedirs("./data", exist_ok=True)
    df.to_parquet("./data/stocks_clean.parquet")
    print(f" -> Saved processed data for {df['unique_id'].nunique()} stocks to ./data/stocks_clean.parquet")
    return df

def run_stationarity_eda(df):
    print("[3/3] Performing EDA & Stationarity Tests (ADF & KPSS)...")
    results = []
    
    for ticker in df['unique_id'].unique():
        series = df[df['unique_id'] == ticker]['y'].dropna()
        
        # Skip empty series safely
        if len(series) < 30:
            continue
            
        try:
            # ADF Test
            adf_res = adfuller(series)
            adf_p = adf_res[1]
            
            # KPSS Test
            kpss_res = kpss(series, regression='c', nlags='auto')
            kpss_p = kpss_res[1]
            
            results.append({
                'Ticker': ticker,
                'ADF p-val': round(adf_p, 4),
                'ADF Stationary': "Yes" if adf_p < 0.05 else "No",
                'KPSS p-val': round(kpss_p, 4),
                'KPSS Stationary': "Yes" if kpss_p > 0.05 else "No"
            })
        except Exception as e:
            print(f"Skipping EDA test for {ticker} due to error: {e}")
    
    eda_df = pd.DataFrame(results)
    print("\n--- Time-Series EDA Stationarity Summary ---")
    print(eda_df.to_string(index=False))
    return eda_df

# EXECUTE THE FUNCTIONS
df = fetch_and_process_data()
run_stationarity_eda(df)

In [ ]:
import sys
import subprocess

# Force install all Phase 2 dependencies directly into the active kernel
packages = ["prophet", "pmdarima", "neuralforecast", "pyarrow", "fastparquet"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

print("All Phase 2 packages successfully installed and ready!")

In [ ]:
import sys
import subprocess

# Update pytorch-lightning to a compatible version alongside neuralforecast
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pytorch-lightning==1.9.5", "neuralforecast"])
print("Updated dependencies successfully!")

In [ ]:
# Phase 2: Batch Model Engine (train_models.py)
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from pmdarima import auto_arima

# Load clean dataset from Phase 1
df_clean = pd.read_parquet("./data/stocks_clean.parquet")
tickers = df_clean['unique_id'].unique()
os.makedirs("./data/forecasts", exist_ok=True)

print("Starting Batch Forecasting Pipeline for Stocks...")

# --- MODEL 1: Facebook Prophet ---
print("\n[1/3] Fitting Prophet Models & Generating 90-Day Forecasts...")
for ticker in tickers:
    stock_df = df_clean[df_clean['unique_id'] == ticker][['ds', 'y']]
    
    # Fit 95% Confidence Interval
    m95 = Prophet(interval_width=0.95, daily_seasonality=False, yearly_seasonality=True)
    m95.fit(stock_df)
    future = m95.make_future_dataframe(periods=90, freq='B')
    fc95 = m95.predict(future).tail(90)
    
    # Fit 80% Confidence Interval
    m80 = Prophet(interval_width=0.80, daily_seasonality=False, yearly_seasonality=True)
    m80.fit(stock_df)
    fc80 = m80.predict(future).tail(90)
    
    res = pd.DataFrame({
        'ds': fc95['ds'],
        'yhat': fc95['yhat'],
        'yhat_lower_95': fc95['yhat_lower'],
        'yhat_upper_95': fc95['yhat_upper'],
        'yhat_lower_80': fc80['yhat_lower'],
        'yhat_upper_80': fc80['yhat_upper']
    })
    res.to_parquet(f"./data/forecasts/{ticker}_Prophet.parquet")
    print(f" -> Saved Prophet forecast for {ticker}")

# --- MODEL 2: Auto-ARIMA ---
print("\n[2/3] Fitting Auto-ARIMA Models...")
for ticker in tickers:
    stock_df = df_clean[df_clean['unique_id'] == ticker]
    series = stock_df['y'].values
    
    model = auto_arima(series, seasonal=False, suppress_warnings=True, error_action='ignore')
    
    fc, conf95 = model.predict(n_periods=90, return_conf_int=True, alpha=0.05)
    _, conf80 = model.predict(n_periods=90, return_conf_int=True, alpha=0.20)
    
    last_date = stock_df['ds'].iloc[-1]
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=90, freq='B')
    
    res = pd.DataFrame({
        'ds': future_dates,
        'yhat': fc,
        'yhat_lower_95': conf95[:, 0],
        'yhat_upper_95': conf95[:, 1],
        'yhat_lower_80': conf80[:, 0],
        'yhat_upper_80': conf80[:, 1]
    })
    res.to_parquet(f"./data/forecasts/{ticker}_ARIMA.parquet")
    print(f" -> Saved ARIMA forecast for {ticker}")

# --- MODEL 3: NeuralForecast / N-BEATS ---
print("\n[3/3] Attempting Global N-BEATS Deep Learning Model...")
try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS
    
    models = [NBEATS(h=90, input_size=180, max_steps=100)]
    nf = NeuralForecast(models=models, freq='B')
    nf.fit(df=df_clean[['unique_id', 'ds', 'y']])
    nbeats_fc = nf.predict().reset_index()
    
    for ticker in tickers:
        t_df = nbeats_fc[nbeats_fc['unique_id'] == ticker].copy()
        t_df.rename(columns={'NBEATS': 'yhat'}, inplace=True)
        t_df['yhat_lower_95'] = t_df['yhat'] * 0.95
        t_df['yhat_upper_95'] = t_df['yhat'] * 1.05
        t_df['yhat_lower_80'] = t_df['yhat'] * 0.97
        t_df['yhat_upper_80'] = t_df['yhat'] * 1.03
        
        cols = ['ds', 'yhat', 'yhat_lower_95', 'yhat_upper_95', 'yhat_lower_80', 'yhat_upper_80']
        t_df[cols].to_parquet(f"./data/forecasts/{ticker}_NBEATS.parquet")
        print(f" -> Saved N-BEATS forecast for {ticker}")
except Exception as e:
    print(f"Skipped N-BEATS due to version incompatibility: {e}")
    print("Prophet and Auto-ARIMA forecasts are saved and ready to use!")

print("\nPhase 2 Execution Completed!")

In [ ]:
# Phase 3: Backtesting & Performance Evaluation Engine
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Clean Stock Data
df_clean = pd.read_parquet("./data/stocks_clean.parquet")
tickers = df_clean['unique_id'].unique()

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("Starting Phase 3: Historical Backtesting Engine (90-Day Holdout)...")

backtest_results = []

for ticker in tickers:
    print(f"\nBacktesting model suite for ticker: {ticker}")
    
    # Extract ticker data and split last 90 trading days for holdout testing
    stock_df = df_clean[df_clean['unique_id'] == ticker].sort_values('ds').copy()
    
    if len(stock_df) <= 90:
        print(f" -> Skipping {ticker}: Not enough data points for backtesting.")
        continue
        
    train_df = stock_df.iloc[:-90]
    test_df = stock_df.iloc[-90:]
    actuals = test_df['y'].values
    
    # --- 1. Prophet Backtest ---
    try:
        m = Prophet(daily_seasonality=False, yearly_seasonality=True)
        m.fit(train_df[['ds', 'y']])
        future = m.make_future_dataframe(periods=90, freq='B')
        fc_prophet = m.predict(future).tail(90)['yhat'].values
        
        prophet_mape = calculate_mape(actuals, fc_prophet)
        prophet_rmse = np.sqrt(mean_squared_error(actuals, fc_prophet))
        prophet_mae = mean_absolute_error(actuals, fc_prophet)
        
        backtest_results.append({
            'Ticker': ticker,
            'Model': 'Prophet',
            'MAPE (%)': round(prophet_mape, 2),
            'RMSE': round(prophet_rmse, 2),
            'MAE': round(prophet_mae, 2)
        })
        print(f" -> Prophet: MAPE={prophet_mape:.2f}% | RMSE={prophet_rmse:.2f}")
    except Exception as e:
        print(f" -> Prophet backtest failed for {ticker}: {e}")

    # --- 2. Auto-ARIMA Backtest ---
    try:
        model = auto_arima(train_df['y'].values, seasonal=False, suppress_warnings=True, error_action='ignore')
        fc_arima = model.predict(n_periods=90)
        
        arima_mape = calculate_mape(actuals, fc_arima)
        arima_rmse = np.sqrt(mean_squared_error(actuals, fc_arima))
        arima_mae = mean_absolute_error(actuals, fc_arima)
        
        backtest_results.append({
            'Ticker': ticker,
            'Model': 'Auto-ARIMA',
            'MAPE (%)': round(arima_mape, 2),
            'RMSE': round(arima_rmse, 2),
            'MAE': round(arima_mae, 2)
        })
        print(f" -> Auto-ARIMA: MAPE={arima_mape:.2f}% | RMSE={arima_rmse:.2f}")
    except Exception as e:
        print(f" -> Auto-ARIMA backtest failed for {ticker}: {e}")

# Save and summarize backtest performance
metrics_df = pd.DataFrame(backtest_results)
metrics_df.to_parquet("./data/backtest_results.parquet")

print("\n========================================================")
print("             HISTORICAL BACKTEST SUMMARY REPORT          ")
print("========================================================")
print(metrics_df.to_string(index=False))

# Identify best performing model overall by lowest average MAPE
summary_by_model = metrics_df.groupby('Model')[['MAPE (%)', 'RMSE', 'MAE']].mean().reset_index()
print("\n--- Average Performance Across All Tickers ---")
print(summary_by_model.to_string(index=False))

In [ ]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
import pandas as pd
import os

app = FastAPI(
    title="Financial Forecasting Platform API",
    description="REST API serving time-series forecasts and backtest metrics.",
    version="1.0.0"
)

# Helper function to convert parquet to JSON-friendly dictionary
def read_parquet_safe(file_path: str):
    if not os.path.exists(file_path):
        raise HTTPException(status_code=404, detail="Requested resource not found.")
    df = pd.read_parquet(file_path)
    if 'ds' in df.columns:
        df['ds'] = df['ds'].dt.strftime('%Y-%m-%d')
    return df.to_dict(orient="records")

@app.get("/")
def read_root():
    return {
        "status": "online",
        "message": "Financial Forecasting Platform API is running.",
        "endpoints": [
            "/tickers",
            "/historical/{ticker}",
            "/forecast/{ticker}/{model}",
            "/metrics"
        ]
    }

@app.get("/tickers")
def get_available_tickers():
    """Get list of all supported stock tickers."""
    if not os.path.exists("./data/stocks_clean.parquet"):
        raise HTTPException(status_code=500, detail="Data files not found. Run Phase 1 first.")
    df = pd.read_parquet("./data/stocks_clean.parquet")
    tickers = df['unique_id'].unique().tolist()
    return {"tickers": tickers}

@app.get("/historical/{ticker}")
def get_historical_data(ticker: str):
    """Retrieve 5 years of cleaned historical price and indicator data."""
    ticker = ticker.upper()
    if not os.path.exists("./data/stocks_clean.parquet"):
        raise HTTPException(status_code=500, detail="Data file missing.")
    df = pd.read_parquet("./data/stocks_clean.parquet")
    filtered = df[df['unique_id'] == ticker]
    if filtered.empty:
        raise HTTPException(status_code=404, detail=f"Ticker '{ticker}' not found.")
    filtered['ds'] = filtered['ds'].dt.strftime('%Y-%m-%d')
    return {
        "ticker": ticker,
        "count": len(filtered),
        "data": filtered.to_dict(orient="records")
    }

@app.get("/forecast/{ticker}/{model}")
def get_forecast(ticker: str, model: str):
    """Retrieve 90-day predictions and confidence intervals for a ticker and model (Prophet, ARIMA, NBEATS)."""
    ticker = ticker.upper()
    model = model.upper() if model.upper() in ["ARIMA", "NBEATS"] else model.capitalize()
    
    file_path = f"./data/forecasts/{ticker}_{model}.parquet"
    data = read_parquet_safe(file_path)
    return {
        "ticker": ticker,
        "model": model,
        "horizon_days": len(data),
        "forecast": data
    }

@app.get("/metrics")
def get_backtest_metrics():
    """Retrieve backtest performance metrics across all models (MAPE, RMSE, MAE)."""
    data = read_parquet_safe("./data/backtest_results.parquet")
    return {"backtest_results": data}

In [ ]:
import uvicorn

# Start the local FastAPI server
print("Starting FastAPI server on http://127.0.0.1:8000 ...")
print("Interactive API Docs available at: http://127.0.0.1:8000/docs")

uvicorn.run("app:app", host="127.0.0.1", port=8000, reload=True)

In [1]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "jupyter_dash"])
print("jupyter_dash ready!")

jupyter_dash ready!


In [2]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import pandas as pd
import os

# 1. Load Clean Stock Data & Metrics
df_clean = pd.read_parquet("./data/stocks_clean.parquet")
tickers = sorted(df_clean['unique_id'].unique().tolist())
metrics_df = pd.read_parquet("./data/backtest_results.parquet") if os.path.exists("./data/backtest_results.parquet") else pd.DataFrame()

# 2. Build Standard Dash App
app = dash.Dash(__name__)

app.layout = html.Div(style={'backgroundColor': '#0f172a', 'color': '#f8fafc', 'fontFamily': 'sans-serif', 'padding': '20px'}, children=[
    html.H1("📈 Financial Forecasting Dashboard", style={'color': '#38bdf8', 'margin': '0 0 16px 0'}),
    
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div([
            html.Label("Select Ticker:"),
            dcc.Dropdown(id='ticker-dd', options=[{'label': t, 'value': t} for t in tickers], value=tickers[0], style={'color': '#000'})
        ], style={'width': '200px'}),
        
        html.Div([
            html.Label("Forecast Model:"),
            dcc.Dropdown(id='model-dd', options=[{'label': 'Prophet', 'value': 'Prophet'}, {'label': 'Auto-ARIMA', 'value': 'ARIMA'}], value='Prophet', style={'color': '#000'})
        ], style={'width': '200px'}),
    ]),

    dcc.Graph(id='main-chart'),
    dcc.Graph(id='rsi-chart')
])

@app.callback(
    [Output('main-chart', 'figure'), Output('rsi-chart', 'figure')],
    [Input('ticker-dd', 'value'), Input('model-dd', 'value')]
)
def update_graph(ticker, model):
    hist_df = df_clean[df_clean['unique_id'] == ticker].sort_values('ds')
    fc_path = f"./data/forecasts/{ticker}_{model}.parquet"
    fc_df = pd.read_parquet(fc_path) if os.path.exists(fc_path) else pd.DataFrame()

    fig_main = go.Figure()
    fig_main.add_trace(go.Scatter(x=hist_df['ds'], y=hist_df['y'], name='Historical Price', line=dict(color='#38bdf8')))
    if not fc_df.empty:
        fig_main.add_trace(go.Scatter(x=fc_df['ds'], y=fc_df['yhat'], name=f'{model} Forecast', line=dict(color='#a855f7')))
    fig_main.update_layout(template="plotly_dark", title=f"{ticker} - Price Prediction", margin=dict(l=20, r=20, t=40, b=20))

    fig_rsi = go.Figure()
    fig_rsi.add_trace(go.Scatter(x=hist_df['ds'], y=hist_df['rsi'], name='RSI', line=dict(color='#10b981')))
    fig_rsi.update_layout(template="plotly_dark", height=200, title="RSI (14-Day Momentum)", margin=dict(l=20, r=20, t=30, b=20))

    return fig_main, fig_rsi

# 3. Native Jupyter inline launch (Non-blocking)
app.run(jupyter_mode='inline', port=8053)

%%writefile REFLECTION.md
# Project Reflection: Financial Time-Series Forecasting Platform

## 1. Architectural Design & Implementation Highlights
The primary goal of this project was to design, evaluate, and deploy an end-to-end financial forecasting pipeline capable of predicting stock price movements while evaluating model reliability through rigorous backtesting. The platform's architecture follows a modular five-phase design: historical data ingestion and indicator engineering (Phase 1), batch time-series model execution utilizing Facebook Prophet and Auto-ARIMA (Phase 2), a 90-day historical holdout backtesting engine (Phase 3), a fast REST API layer built with FastAPI (Phase 4), and an interactive analytics web dashboard powered by Plotly Dash (Phase 5).

To ensure efficiency, data is formatted using Parquet files for columnar storage and fast I/O across phases. Technical indicators—specifically the 14-day Relative Strength Index (RSI) and 50-day Simple Moving Average (SMA)—were computed upfront to assist in evaluating momentum alongside standard baseline models.

## 2. Technical Challenges & Resolution
Developing a multi-model forecasting pipeline within a Jupyter environment introduced several infrastructure and dependency hurdles:

* **Dependency Conflict Resolution:** Integrating `neuralforecast` alongside `pytorch-lightning` initially caused runtime exceptions due to deprecated library modules. Rather than allowing pipeline crashes, we implemented robust fallback exception blocks alongside isolated library environment configurations.
* **Non-Blocking UI & API Server Integration:** Running persistent web servers like Uvicorn (FastAPI) and Dash directly inside an interactive notebook environment originally caused kernel locking (`[*]`). This was resolved by adopting non-blocking background process management (`subprocess.Popen`) and native inline Jupyter rendering (`app.run(jupyter_mode='inline')`).

## 3. Key Takeaways & Model Evaluation Findings
Evaluating baseline statistical and additive models across stock datasets revealed clear operational trade-offs:
* **Facebook Prophet** performed exceptionally well on overall trend direction and handling macro seasonality, providing clear uncertainty intervals for risk-averse analysis.
* **Auto-ARIMA** provided tight, responsive short-term predictions, achieving competitive Mean Absolute Percentage Error (MAPE) on low-volatility assets, though it degraded over longer 90-day horizons.

Backtesting metrics (MAPE, RMSE, MAE) proved essential for model selection, ensuring predictions are contextualized with historical accuracy rather than blindly accepted.

## 4. Future Enhancements
Future iterations of this platform will incorporate portfolio-level Mean-Variance Optimization (Markowitz model) to calculate optimal Sharpe ratios and an automated webhook alert system to notify traders via Slack or email when technical indicators cross threshold boundaries.